## 🏢 03 — MCP Integration & Agent as Tool

**Goal**: Connect agents to enterprise systems using **Hosted MCP tools**, **FoundryAgent** (service-managed agents), and **agent-as-tool** composition.

- `client.get_mcp_tool()` — hosted MCP server integration
- `agent.as_tool()` — use one agent as a tool for another

MCP (Model Context Protocol) servers are specialized programs that let AI models securely connect to external data sources and tools through a standardized protocol

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from utils.config import get_foundry_client, get_credential, get_env
from agent_framework import Agent, tool

client = get_foundry_client()
print("✅ Connected to Foundry")

✅ Connected to Foundry


---
### Part A — Hosted MCP Tools 🌐
Connect to a hosted MCP server (e.g. Microsoft Learn) managed by Foundry.

In [2]:
# ── A1. Create agent with MCP tool ───────────────────────────────────────────
MCP_URL = get_env("MCP_SERVER_URL", required=False)

if not MCP_URL or MCP_URL.startswith("<"):
    print("⚠️  MCP_SERVER_URL not set — skipping MCP demo.")
else:
    learn_mcp = client.get_mcp_tool(
        name="Microsoft Learn MCP",
        url=MCP_URL,
        approval_mode="never_require",
    )

    mcp_agent = Agent(
        client=client,
        name="MicrosoftLearnAgent",
        instructions=(
            "You are a technical documentation assistant. Use the MCP tool to "
            "search Microsoft Learn and provide accurate, up-to-date guidance "
            "with links to official docs."
        ),
        tools=learn_mcp,
    )
    print(f"🤖 MCP agent: {mcp_agent.name}")

🤖 MCP agent: MicrosoftLearnAgent


In [3]:
# ── A2. Ask a documentation question ─────────────────────────────────────────
if MCP_URL and not MCP_URL.startswith("<"):
    result = await mcp_agent.run(
        "How do I create an Azure Function with a timer trigger in Python?"
    )
    print(f"🗨️  Agent: {result}")

🗨️  Agent: To create an Azure Function with a timer trigger in Python, follow these steps:

### 1. Configure the Timer Trigger

Create a `function.json` file with the following configuration:

```json
{
  "schedule": "0 */5 * * * *",
  "name": "myTimer",
  "type": "timerTrigger",
  "direction": "in"
}
```

**Explanation:**
- **`schedule`**: Uses NCRONTAB expression. In this example, it triggers every 5 minutes.
- **`name`**: The variable name for the timer object.
- **`type`**: Must be set to `"timerTrigger"`.
- **`direction`**: Set to `"in"`.

### 2. Write the Function Code

Here's an example Python function:

```python
import datetime
import logging
import azure.functions as func

def main(mytimer: func.TimerRequest) -> None:
    utc_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()

    if mytimer.past_due:
        logging.info('The timer is past due!')

    logging.info('Python timer trigger function ran at %s', utc_timestamp)
```

**Explanation:**
- **`mytimer: 

---
### Part B — Agent-as-Tool Composition 🔗
Use one agent as a tool for another — enabling agent delegation without a full workflow.

In [4]:
# ── C1. Create specialist agents ─────────────────────────────────────────────
from typing import Annotated
from pydantic import Field

# A "math expert" agent
math_agent = client.as_agent(
    name="MathExpert",
    description="An agent that solves math problems step by step.",
    instructions="You are a math expert. Solve problems step by step with clear explanations.",
    tools=client.get_code_interpreter_tool(),
)

# A custom function tool for CRM lookup
@tool(description="Look up customer info by email.")
def lookup_customer(
    email: Annotated[str, Field(description="Customer email address")],
) -> str:
    customers = {
        "alice@contoso.com": {"name": "Alice Smith", "plan": "Enterprise", "tickets": 3},
        "bob@contoso.com": {"name": "Bob Jones", "plan": "Starter", "tickets": 12},
    }
    info = customers.get(email.lower())
    if not info:
        return f"No customer found with email: {email}"
    return str(info)

print("🔬 Specialist agents and tools created")

🔬 Specialist agents and tools created


In [5]:
# ── C2. Create coordinator that uses specialists as tools ────────────────────
coordinator = Agent(
    client=client,
    name="Coordinator",
    instructions=(
        "You are a helpful coordinator. You have access to:\n"
        "- A math expert for calculations and data analysis\n"
        "- A customer lookup tool for CRM queries\n"
        "Route requests to the appropriate tool and present unified answers."
    ),
    tools=[
        math_agent.as_tool(),       # agent used as a tool!
        lookup_customer,            # function tool
    ],
)
print(f"🎯 Coordinator agent: {coordinator.name}")
print("   Tools: MathExpert (agent-as-tool), lookup_customer (function tool)")

🎯 Coordinator agent: Coordinator
   Tools: MathExpert (agent-as-tool), lookup_customer (function tool)


In [6]:
# ── C3. Test routing — math question delegates to MathExpert ─────────────────
result = await coordinator.run(
    "What is the compound annual growth rate if revenue grew from $1M to $2.5M over 3 years?"
)
print(f"🗨️  Coordinator: {result}")

🗨️  Coordinator: The Compound Annual Growth Rate (CAGR) for the revenue growth from $1,000,000 to $2,500,000 over 3 years is approximately 35.72%.


In [7]:
# ── C4. Test routing — CRM query uses function tool ──────────────────────────
result = await coordinator.run(
    "Look up the customer info for bob@contoso.com and tell me about their plan."
)
print(f"🗨️  Coordinator: {result}")

🗨️  Coordinator: Bob Jones is on the "Starter" plan and currently has 12 tickets available.


---
## ✅ What you demonstrated
| Capability | Agent Framework Feature |
|---|---|
| Hosted MCP integration | `client.get_mcp_tool(name=..., url=...)` |
| Agent-as-tool | `agent.as_tool()` — one agent calls another |
| Mixed tool types | Function tools + agent tools on same coordinator |

**Next →** [04_multi_agent_orchestration.ipynb](./04_multi_agent_orchestration.ipynb) — graph-based Workflows with `WorkflowBuilder`.